# LeetCode 475 — Heaters bằng Python

Link đề: https://leetcode.com/problems/heaters/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers (hai con trỏ), Binary Search (tìm kiếm nhị phân, cả "nhị phân trên đáp án"), Sorting

Skeleton LeetCode:

```python
class Solution:
    def findRadius(self, houses: List[int], heaters: List[int]) -> int:
        ...
```

Ghi chú: bài này **không** phải Premium. Đây là bài rất đẹp để học 3 kỹ thuật cùng lúc trên một đề: tìm "phần tử gần nhất" bằng `bisect`, **hai con trỏ** trên hai mảng đã sắp xếp, và **nhị phân trên đáp án** (đáp án càng lớn càng dễ thỏa).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **một bán kính chung nhỏ nhất** `r` sao cho mọi ngôi nhà đều nằm trong khoảng `[h - r, h + r]` của **ít nhất một** lò sưởi `h`; hiểu công thức đáp án = **max** (qua các nhà) của **min** (qua các lò) khoảng cách.
2. Dùng các công cụ Python: `sort`, `abs`, `max`/`min` với generator, `bisect.bisect_left`, và con trỏ chỉ số trên hai mảng đã sắp xếp.
3. Nhận ra hai ý tưởng đột phá: **(a)** mỗi nhà chỉ cần xét lò **gần nhất** (một bên trái, một bên phải) tìm bằng nhị phân; **(b)** khả năng "phủ hết nhà" là **đơn điệu** theo `r`, nên nhị phân được trên chính `r`.
4. Cài đặt 4 cách khác nhau: **vét cạn `O(M * H)` → sắp xếp + `bisect` cho từng nhà → nhị phân trên bán kính + kiểm tra hai con trỏ → hai con trỏ trực tiếp (merge-like)**, và so sánh độ phức tạp.
5. Tránh các lỗi kinh điển: quên sắp xếp lò sưởi, chỉ xét một phía của `bisect`, lấy `min` thay vì `max` trên các nhà, đặt cận trên nhị phân quá nhỏ, và quên xử lý nhà nằm ngoài cùng (trước lò đầu / sau lò cuối).

## 2. Đề bài & ví dụ

Mùa đông đến rồi! Nhiệm vụ của bạn là thiết kế một **máy sưởi tiêu chuẩn** có **bán kính sưởi cố định** để sưởi ấm tất cả các ngôi nhà.

Một ngôi nhà được sưởi ấm nếu nó nằm trong **bán kính sưởi** của một lò (tức khoảng cách từ nhà đến lò `<= r`). Cho vị trí các ngôi nhà `houses` và các lò sưởi `heaters` trên một **đường thẳng ngang**, trả về **bán kính nhỏ nhất** sao cho các lò sưởi phủ được **mọi** ngôi nhà.

**Chú ý:** mọi lò sưởi đều dùng **chung** một bán kính.

**Ví dụ 1:** `houses = [1,2,3]`, `heaters = [2]` → `1`. Lò duy nhất ở vị trí `2`; bán kính `1` phủ `[1, 3]`.

**Ví dụ 2:** `houses = [1,2,3,4]`, `heaters = [1,4]` → `1`. Lò ở `1` phủ nhà `1, 2`; lò ở `4` phủ nhà `3, 4`.

**Ví dụ 3:** `houses = [1,5]`, `heaters = [2]` → `3`. Nhà `5` cách lò `2` đúng `3`.

**Ràng buộc:**

- `1 <= houses.length, heaters.length <= 3 * 10^4`
- `1 <= houses[i], heaters[i] <= 10^9`

Với `3 × 10^4` nhà và lò, vét cạn `O(M * H) = 9 × 10^8` là quá chậm; cần `O((M + H) log)` hoặc tuyến tính sau sắp xếp.

In [1]:
import bisect
import random
import time
from typing import List

examples = [([1, 2, 3], [2], 1), ([1, 2, 3, 4], [1, 4], 1), ([1, 5], [2], 3)]
for houses, heaters, ans in examples:
    print(f"houses = {houses}, heaters = {heaters}  -> {ans}")

houses = [1, 2, 3], heaters = [2]  -> 1
houses = [1, 2, 3, 4], heaters = [1, 4]  -> 1
houses = [1, 5], heaters = [2]  -> 3


## 3. Kiến thức nền

### 3.1. Khoảng cách trên đường thẳng, `abs`, `max`/`min` với generator

Khoảng cách giữa nhà `x` và lò `y` là `abs(x - y)`. Để biết nhà `x` cách lò **gần nhất** bao xa, lấy `min` qua mọi lò. Dùng **generator** (biểu thức trong ngoặc, không dựng list) cho gọn.

In [2]:
x = 5
heaters = [2, 9, 6]
print([abs(x - h) for h in heaters])                  # [3, 4, 1]
print(min(abs(x - h) for h in heaters))               # 1 : lò gần nhất cách 1
houses = [1, 5]
print(max(min(abs(x - h) for h in [2]) for x in houses))   # 3 : nhà "xa" nhất quyết định (ví dụ 3)

[3, 4, 1]
1
3


### 3.2. `sort` và `sorted`

`list.sort()` sắp xếp **tại chỗ** (đổi luôn list gốc, trả `None`); `sorted(list)` trả **list mới** và giữ nguyên gốc. Trên LeetCode đổi mảng đầu vào thường chấp nhận được, nhưng khi tự test nhiều cách cùng một input, hãy truyền **bản sao** để các cách không ảnh hưởng nhau.

In [3]:
a = [3, 1, 2]
b = sorted(a)
print(a, b)          # [3, 1, 2] [1, 2, 3]
r = a.sort()
print(a, r)          # [1, 2, 3] None

[3, 1, 2] [1, 2, 3]
[1, 2, 3] None


### 3.3. `bisect.bisect_left`: tìm chỗ chèn trong mảng đã sắp xếp

`bisect.bisect_left(arr, x)` trả về chỉ số `i` nhỏ nhất sao cho `arr[i] >= x` (hoặc `len(arr)` nếu mọi phần tử `< x`). Mảng **phải đã sắp xếp**. Khi đó: `arr[i]` là phần tử **gần nhất bên phải** (`>= x`), và `arr[i - 1]` là phần tử **gần nhất bên trái** (`< x`).

In [4]:
heaters = [1, 4, 9]
for x in (0, 1, 3, 4, 6, 10):
    i = bisect.bisect_left(heaters, x)
    phai = heaters[i] if i < len(heaters) else None
    trai = heaters[i - 1] if i > 0 else None
    print(f"x = {x:2}: i = {i}, trai = {trai}, phai = {phai}")

x =  0: i = 0, trai = None, phai = 1
x =  1: i = 0, trai = None, phai = 1
x =  3: i = 1, trai = 1, phai = 4
x =  4: i = 1, trai = 1, phai = 4
x =  6: i = 2, trai = 4, phai = 9
x = 10: i = 3, trai = 9, phai = None


### 3.4. Nhị phân trên đáp án (binary search on answer)

Khi một câu hỏi "với `r` này có được không?" có tính **đơn điệu** (nếu `r` được thì `r + 1` cũng được), ta tìm `r` **nhỏ nhất được** bằng nhị phân: `left` là cận dưới, `right` là cận trên "chắc chắn được"; lặp `mid`; nếu `mid` được thì `right = mid`, ngược lại `left = mid + 1`; kết thúc khi `left == right`.

In [5]:
def nho_nhat_duoc(duoc, lo, hi):
    """Tìm giá trị nhỏ nhất trong [lo, hi] mà duoc(v) = True (giả sử duoc(hi) = True và đơn điệu)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if duoc(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(nho_nhat_duoc(lambda v: v * v >= 50, 0, 100))      # 8 (8*8 = 64 >= 50, 7*7 = 49 < 50)

8


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 3:** `houses = [1, 5]`, `heaters = [2]`.

- Nhà `1`: cách lò `2` là `1`. Lò gần nhất cách `1`.
- Nhà `5`: cách lò `2` là `3`. Lò gần nhất cách `3`.
- Muốn **mọi** nhà được sưởi, `r` phải `>=` khoảng cách gần nhất của **từng** nhà, tức `r >= max(1, 3) = 3`.

**Ví dụ 2:** `houses = [1,2,3,4]`, `heaters = [1,4]`. Khoảng cách tới lò gần nhất: nhà `1` → `0`; nhà `2` → `min(1, 2) = 1`; nhà `3` → `min(2, 1) = 1`; nhà `4` → `0`. Lấy `max` → `1`.

**Công thức:**

```
đáp án = max over houses x  of  ( min over heaters h  of  |x - h| )
```

Vì sao `max` bên ngoài, `min` bên trong? Mỗi nhà chỉ cần **một** lò phủ nó (nên chọn lò gần nhất: `min`); nhưng **tất cả** nhà đều phải được phủ (nên `r` phải thỏa nhà "khó" nhất: `max`).

**Câu hỏi:** làm sao tìm "lò gần nhất của mỗi nhà" nhanh hơn việc xét hết mọi lò?

In [6]:
# Tính tay bằng máy: bảng khoảng cách nhà-lò cho ví dụ 2
houses, heaters = [1, 2, 3, 4], [1, 4]
print("nha | khoang cach toi tung lo | gan nhat")
for x in houses:
    d = [abs(x - h) for h in heaters]
    print(f"{x:3} | {d} | {min(d)}")
print("dap an =", max(min(abs(x - h) for h in heaters) for x in houses))

nha | khoang cach toi tung lo | gan nhat
  1 | [0, 3] | 0
  2 | [1, 2] | 1
  3 | [2, 1] | 1
  4 | [3, 0] | 0
dap an = 1


## 5. Cách 1 — Brute Force / baseline: với mỗi nhà, xét mọi lò

**Ý tưởng:** đúng công thức mục 4. Với mỗi nhà tính `min` khoảng cách qua **tất cả** lò, rồi lấy `max` qua các nhà.

In [7]:
def find_radius_brute(houses, heaters):
    kq = 0
    for x in houses:
        gan_nhat = min(abs(x - h) for h in heaters)
        kq = max(kq, gan_nhat)
    return kq

for houses, heaters, ans in examples:
    print(find_radius_brute(houses, heaters), "(mong doi", ans, ")")

1 (mong doi 1 )
1 (mong doi 1 )
3 (mong doi 3 )


**Độ phức tạp:** Time `O(M * H)` (`M` nhà, `H` lò), Space `O(1)`.

**Nút thắt:** với `M = H = 3 × 10^4` là `9 × 10^8` phép tính: quá chậm. Ta đang **bỏ phí** thông tin vị trí: lò xa tít bên kia chắc chắn không phải gần nhất. Nếu lò đã sắp xếp, lò gần nhất của một nhà nằm ngay **cạnh** vị trí chèn của nhà đó. Hãy đo thời gian để thấy `O(M * H)` tăng thế nào.

In [8]:
rng = random.Random(1)
for size in (200, 400, 800, 1600):
    houses = [rng.randint(1, 10**9) for _ in range(size)]
    heaters = [rng.randint(1, 10**9) for _ in range(size)]
    t0 = time.perf_counter()
    find_radius_brute(houses, heaters)
    print(f"M = H = {size:4}: {size * size:8} cap, mat {time.perf_counter() - t0:.3f}s")

M = H =  200:    40000 cap, mat 0.002s
M = H =  400:   160000 cap, mat 0.010s
M = H =  800:   640000 cap, mat 0.036s


M = H = 1600:  2560000 cap, mat 0.147s


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1: lò gần nhất của một nhà nằm cạnh vị trí chèn.** Sắp xếp `heaters`. Với nhà `x`, gọi `i = bisect_left(heaters, x)`. Lò gần nhất **bên phải** (`>= x`) là `heaters[i]` (nếu `i < H`), lò gần nhất **bên trái** (`< x`) là `heaters[i - 1]` (nếu `i > 0`). Lò gần nhất là một trong hai, nên chỉ so **hai** ứng viên thay vì `H`.

**Bước đột phá 2: đơn điệu theo `r`.** Nếu bán kính `r` phủ được mọi nhà thì `r + 1` cũng phủ được. Vậy ta có thể **nhị phân trên `r`** và kiểm tra "`r` có đủ không" trong `O(M + H)` bằng hai con trỏ trên hai mảng đã sắp xếp.

**Bước đột phá 3: hai con trỏ trực tiếp.** Nếu cả nhà lẫn lò đều đã sắp xếp, thì lò gần nhất của các nhà **di chuyển đơn điệu** sang phải: ta chỉ cần một con trỏ `j` trên lò, tiến khi lò kế tiếp **gần hơn hoặc bằng**. Không cần nhị phân nữa: tổng cộng `O(M + H)` sau sắp xếp.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp lò + `bisect` cho từng nhà

**Giải thích từng đoạn:**

1. `heaters.sort()` (đưa về thứ tự tăng dần).
2. Với mỗi nhà `x`: `i = bisect_left(heaters, x)`.
3. Khoảng cách tới lò phải: `heaters[i] - x` nếu `i < H`, ngược lại coi là vô cùng (`inf`).
4. Khoảng cách tới lò trái: `x - heaters[i - 1]` nếu `i > 0`, ngược lại `inf`.
5. Khoảng cách gần nhất của nhà `= min` hai giá trị; đáp án `= max` qua các nhà.

Luôn có ít nhất một phía hợp lệ vì `H >= 1`.

In [9]:
def find_radius_bisect(houses, heaters):
    heaters = sorted(heaters)
    H = len(heaters)
    INF = float("inf")
    kq = 0
    for x in houses:
        i = bisect.bisect_left(heaters, x)
        phai = heaters[i] - x if i < H else INF
        trai = x - heaters[i - 1] if i > 0 else INF
        kq = max(kq, min(phai, trai))
    return kq

for houses, heaters, ans in examples:
    print(find_radius_bisect(houses, heaters), "(mong doi", ans, ")")

1 (mong doi 1 )
1 (mong doi 1 )
3 (mong doi 3 )


**Code nộp LeetCode (Cách 2):**

```python
import bisect

class Solution:
    def findRadius(self, houses: List[int], heaters: List[int]) -> int:
        heaters.sort()
        H = len(heaters)
        INF = float("inf")
        kq = 0
        for x in houses:
            i = bisect.bisect_left(heaters, x)
            phai = heaters[i] - x if i < H else INF
            trai = x - heaters[i - 1] if i > 0 else INF
            kq = max(kq, min(phai, trai))
        return kq
```

**Độ phức tạp:** Time `O(H log H + M log H)`, Space `O(1)` thêm (không kể sắp xếp). Nhà **không cần** sắp xếp ở cách này.

### Cách 3 — Nhị phân trên bán kính + kiểm tra bằng hai con trỏ

**Ý tưởng:** hàm `duoc(r)`: "bán kính `r` có phủ hết nhà không?". Đơn điệu theo `r` (mục 6). Tìm `r` nhỏ nhất thỏa `duoc(r)`.

**Hàm `duoc(r)` bằng hai con trỏ** (nhà `houses` và lò `heaters` đều đã sắp xếp):

- `i` là nhà đang xét, `j` là lò đang xét.
- Lò `j` phủ đoạn `[heaters[j] - r, heaters[j] + r]`.
- Nếu `houses[i] < heaters[j] - r`: nhà nằm **bên trái** vùng phủ của lò `j` và (vì mọi lò từ `j` trở đi đều ở bên phải, còn các lò trước đã bỏ qua) **không còn lò nào phủ được** → `False`.
- Nếu `houses[i] > heaters[j] + r`: nhà nằm **bên phải** vùng phủ lò `j` → thử lò kế tiếp `j += 1`.
- Ngược lại nhà nằm trong vùng phủ → `i += 1`.
- Nếu hết lò mà còn nhà → `False`; hết nhà → `True`.

**Cận cho nhị phân:** `left = 0`, `right = 10^9` (khoảng cách tối đa giữa hai vị trí trong `[1, 10^9]` là `10^9 - 1 <= 10^9`, nên `right = 10^9` luôn được).

In [10]:
def find_radius_binary(houses, heaters):
    houses = sorted(houses)
    heaters = sorted(heaters)
    M, H = len(houses), len(heaters)

    def duoc(r):
        i = j = 0
        while i < M:
            if j >= H:
                return False                       # hết lò mà còn nhà chưa phủ
            lo, hi = heaters[j] - r, heaters[j] + r
            if houses[i] < lo:
                return False                       # nhà nằm trước vùng phủ lò j
            if houses[i] > hi:
                j += 1                             # lò j quá xa bên trái, thử lò kế
            else:
                i += 1                             # nhà i được phủ
        return True

    left, right = 0, 10**9
    while left < right:
        mid = (left + right) // 2
        if duoc(mid):
            right = mid
        else:
            left = mid + 1
    return left

for houses, heaters, ans in examples:
    print(find_radius_binary(houses, heaters), "(mong doi", ans, ")")

1 (mong doi 1 )
1 (mong doi 1 )
3 (mong doi 3 )


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def findRadius(self, houses: List[int], heaters: List[int]) -> int:
        houses.sort()
        heaters.sort()
        M, H = len(houses), len(heaters)

        def duoc(r):
            i = j = 0
            while i < M:
                if j >= H:
                    return False
                lo, hi = heaters[j] - r, heaters[j] + r
                if houses[i] < lo:
                    return False
                if houses[i] > hi:
                    j += 1
                else:
                    i += 1
            return True

        left, right = 0, 10**9
        while left < right:
            mid = (left + right) // 2
            if duoc(mid):
                right = mid
            else:
                left = mid + 1
        return left
```

**Độ phức tạp:** Time `O(M log M + H log H + (M + H) log R)` với `R = 10^9` (khoảng `30` lần nhị phân), Space `O(1)` thêm.

### Cách 4 — Hai con trỏ trực tiếp (merge-like), không nhị phân

**Ý tưởng:** sắp xếp cả hai mảng. Duyệt các nhà từ trái sang phải; giữ con trỏ lò `j`. Với nhà `x`, **tiến `j`** chừng nào lò kế tiếp `heaters[j + 1]` **gần `x` hơn hoặc bằng** lò hiện tại: `abs(heaters[j + 1] - x) <= abs(heaters[j] - x)`. Khi dừng, `heaters[j]` là lò gần nhất của `x`. Vì nhà tăng dần, lò gần nhất cũng chỉ tiến lên (không lùi) nên `j` không bao giờ cần lùi lại.

**Vì sao điều kiện tiến là đúng?** Khoảng cách `|heaters[j] - x|` theo `j` có dạng chữ V (giảm rồi tăng) vì `heaters` đã sắp xếp; ta dừng ở điểm đáy.

In [11]:
def find_radius_two_pointers(houses, heaters):
    houses = sorted(houses)
    heaters = sorted(heaters)
    H = len(heaters)
    j = 0
    kq = 0
    for x in houses:
        while j + 1 < H and abs(heaters[j + 1] - x) <= abs(heaters[j] - x):
            j += 1
        kq = max(kq, abs(heaters[j] - x))
    return kq

for houses, heaters, ans in examples:
    print(find_radius_two_pointers(houses, heaters), "(mong doi", ans, ")")

1 (mong doi 1 )
1 (mong doi 1 )
3 (mong doi 3 )


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def findRadius(self, houses: List[int], heaters: List[int]) -> int:
        houses.sort()
        heaters.sort()
        H = len(heaters)
        j = 0
        kq = 0
        for x in houses:
            while j + 1 < H and abs(heaters[j + 1] - x) <= abs(heaters[j] - x):
                j += 1
            kq = max(kq, abs(heaters[j] - x))
        return kq
```

**Độ phức tạp:** Time `O(M log M + H log H)` cho sắp xếp, phần quét chỉ `O(M + H)`; Space `O(1)` thêm.

## 8. TRACE từng bước

### 8.1. Cách 2 (bisect) trên Ví dụ 2: `houses = [1,2,3,4]`, `heaters = [1,4]`

| nhà `x` | `i = bisect_left` | lò trái | lò phải | khoảng cách trái | khoảng cách phải | gần nhất | `kq` sau |
|---------|-------------------|---------|---------|------------------|------------------|----------|----------|
| 1 | 0 | không | 1 | `inf` | 0 | 0 | 0 |
| 2 | 1 | 1 | 4 | 1 | 2 | 1 | 1 |
| 3 | 1 | 1 | 4 | 2 | 1 | 1 | 1 |
| 4 | 1 | 1 | 4 | 3 | 0 | 0 | 1 |

In [12]:
def trace_bisect(houses, heaters):
    heaters = sorted(heaters)
    H = len(heaters)
    INF = float("inf")
    kq = 0
    print("heaters da sap xep:", heaters)
    for x in houses:
        i = bisect.bisect_left(heaters, x)
        phai = heaters[i] - x if i < H else INF
        trai = x - heaters[i - 1] if i > 0 else INF
        gan = min(phai, trai)
        kq = max(kq, gan)
        lo_trai = heaters[i - 1] if i > 0 else None
        lo_phai = heaters[i] if i < H else None
        print(f"nha {x}: i = {i}, lo trai = {lo_trai}, lo phai = {lo_phai}, d_trai = {trai}, d_phai = {phai}, gan nhat = {gan}, kq = {kq}")
    return kq

print("KET QUA:", trace_bisect([1, 2, 3, 4], [1, 4]))

heaters da sap xep: [1, 4]
nha 1: i = 0, lo trai = None, lo phai = 1, d_trai = inf, d_phai = 0, gan nhat = 0, kq = 0
nha 2: i = 1, lo trai = 1, lo phai = 4, d_trai = 1, d_phai = 2, gan nhat = 1, kq = 1
nha 3: i = 1, lo trai = 1, lo phai = 4, d_trai = 2, d_phai = 1, gan nhat = 1, kq = 1
nha 4: i = 1, lo trai = 1, lo phai = 4, d_trai = 3, d_phai = 0, gan nhat = 0, kq = 1
KET QUA: 1


### 8.2. Cách 3: nhị phân trên `r` và kiểm tra hai con trỏ (Ví dụ 3: `[1,5]`, `[2]`)

Ta in từng bước `mid` của nhị phân; để dễ đọc, dùng cận `right = 8` thay vì `10^9` (vẫn đủ vì đáp án `3`).

In [13]:
def trace_binary(houses, heaters, right0=8):
    houses = sorted(houses)
    heaters = sorted(heaters)
    M, H = len(houses), len(heaters)

    def duoc(r, noi_chuyen=True):
        i = j = 0
        while i < M:
            if j >= H:
                if noi_chuyen: print(f"    het lo, nha {houses[i]} chua phu -> False")
                return False
            lo, hi = heaters[j] - r, heaters[j] + r
            if houses[i] < lo:
                if noi_chuyen: print(f"    nha {houses[i]} < {lo} (truoc vung phu lo {heaters[j]}) -> False")
                return False
            if houses[i] > hi:
                if noi_chuyen: print(f"    nha {houses[i]} > {hi} (sau vung phu lo {heaters[j]}) -> chuyen lo")
                j += 1
            else:
                if noi_chuyen: print(f"    nha {houses[i]} trong [{lo}, {hi}] cua lo {heaters[j]} -> phu")
                i += 1
        return True

    left, right = 0, right0
    while left < right:
        mid = (left + right) // 2
        print(f"left = {left}, right = {right}, mid = {mid}")
        ok = duoc(mid)
        print(f"  duoc({mid}) = {ok}")
        if ok:
            right = mid
        else:
            left = mid + 1
    print("KET QUA r nho nhat =", left)
    return left

trace_binary([1, 5], [2])

left = 0, right = 8, mid = 4
    nha 1 trong [-2, 6] cua lo 2 -> phu
    nha 5 trong [-2, 6] cua lo 2 -> phu
  duoc(4) = True
left = 0, right = 4, mid = 2
    nha 1 trong [0, 4] cua lo 2 -> phu
    nha 5 > 4 (sau vung phu lo 2) -> chuyen lo
    het lo, nha 5 chua phu -> False
  duoc(2) = False
left = 3, right = 4, mid = 3
    nha 1 trong [-1, 5] cua lo 2 -> phu
    nha 5 trong [-1, 5] cua lo 2 -> phu
  duoc(3) = True
KET QUA r nho nhat = 3


3

### 8.3. Cách 4: hai con trỏ trên Ví dụ 1 mở rộng

Ta chọn `houses = [1, 3, 6, 10]`, `heaters = [2, 7]` để thấy con trỏ lò `j` tiến một lần.

In [14]:
def trace_two_pointers(houses, heaters):
    houses = sorted(houses)
    heaters = sorted(heaters)
    H = len(heaters)
    j = 0
    kq = 0
    for x in houses:
        while j + 1 < H and abs(heaters[j + 1] - x) <= abs(heaters[j] - x):
            print(f"  nha {x}: lo {heaters[j + 1]} (d={abs(heaters[j + 1] - x)}) <= lo {heaters[j]} (d={abs(heaters[j] - x)}) -> j += 1")
            j += 1
        d = abs(heaters[j] - x)
        kq = max(kq, d)
        print(f"nha {x}: lo gan nhat = {heaters[j]} (j = {j}), d = {d}, kq = {kq}")
    return kq

print("KET QUA:", trace_two_pointers([1, 3, 6, 10], [2, 7]))

nha 1: lo gan nhat = 2 (j = 0), d = 1, kq = 1
nha 3: lo gan nhat = 2 (j = 0), d = 1, kq = 1
  nha 6: lo 7 (d=1) <= lo 2 (d=4) -> j += 1
nha 6: lo gan nhat = 7 (j = 1), d = 1, kq = 1
nha 10: lo gan nhat = 7 (j = 1), d = 3, kq = 3
KET QUA: 3


## 9. Code nộp LeetCode

Ta nộp **Cách 4** (hai con trỏ): ngắn, không cần nhị phân. Cách 2 và 3 cũng đúng, và cách 2 không đòi hỏi nhà phải sắp xếp.

```python
class Solution:
    def findRadius(self, houses: List[int], heaters: List[int]) -> int:
        houses.sort()
        heaters.sort()
        H = len(heaters)
        j = 0
        kq = 0
        for x in houses:
            while j + 1 < H and abs(heaters[j + 1] - x) <= abs(heaters[j] - x):
                j += 1
            kq = max(kq, abs(heaters[j] - x))
        return kq
```

In [15]:
class Solution:
    def findRadius(self, houses: List[int], heaters: List[int]) -> int:
        houses.sort()
        heaters.sort()
        H = len(heaters)
        j = 0
        kq = 0
        for x in houses:
            while j + 1 < H and abs(heaters[j + 1] - x) <= abs(heaters[j] - x):
                j += 1
            kq = max(kq, abs(heaters[j] - x))
        return kq

print(Solution().findRadius([1, 2, 3], [2]))          # 1
print(Solution().findRadius([1, 2, 3, 4], [1, 4]))    # 1
print(Solution().findRadius([1, 5], [2]))             # 3

1
1
3


## 10. Vì sao thuật toán đúng

**Công thức gốc.** Bán kính `r` phủ hết nhà khi và chỉ khi mỗi nhà `x` có một lò `h` với `|x - h| <= r`, tức `min_h |x - h| <= r` với mọi `x`, tức `r >= max_x min_h |x - h|`. Bán kính nhỏ nhất chính là vế phải.

**Cách 2 (bisect).** Với `heaters` đã sắp xếp, các lò `< x` đều nằm bên trái và lò gần nhất trong số đó là phần tử lớn nhất `< x` (`heaters[i - 1]`); các lò `>= x` đều nằm bên phải và gần nhất là phần tử nhỏ nhất `>= x` (`heaters[i]`). Lò gần nhất tổng thể là một trong hai.

**Cách 3 (nhị phân).** Đơn điệu: nếu lò `h` phủ nhà `x` với `r` thì cũng phủ với `r + 1`. Hàm `duoc` đúng vì: (i) nếu nhà `houses[i] < heaters[j] - r` thì mọi lò `j' >= j` đều còn lớn hơn nên cũng không phủ được nhà đó; còn các lò `j' < j` đã bị bỏ qua vì có một nhà `<= houses[i]` nằm bên phải vùng phủ của chúng, nên nhà hiện tại (lớn hơn hoặc bằng) cũng nằm bên phải vùng phủ ấy. Vậy cả hai phía đều không còn lò nào phủ được nhà này và trả `False` là đúng; bỏ lò `j'` là an toàn. (ii) Nhà trong vùng phủ `[heaters[j] - r, heaters[j] + r]` thì phủ được.

**Cách 4 (hai con trỏ).** Hàm `j -> |heaters[j] - x|` với `heaters` tăng dần là **không tăng** rồi **không giảm** (dạng chữ V). Điều kiện `|heaters[j+1] - x| <= |heaters[j] - x|` còn đúng khi ta chưa tới đáy, nên `while` dừng tại một vị trí đáy, tức là một lò gần nhất. Khi `x` tăng, đáy chỉ dịch sang phải, nên không cần lùi `j`.

Hai con trỏ `j` chỉ tiến, tổng số bước tăng `j` không quá `H`.

## 11. Phân tích Time / Space Complexity

`M` nhà, `H` lò.

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Vét cạn | `O(M * H)` | `O(1)` | oracle |
| 2. Sắp xếp lò + bisect | `O(H log H + M log H)` | `O(1)` thêm | nhà không cần sắp xếp |
| 3. Nhị phân trên `r` + hai con trỏ | `O(M log M + H log H + (M + H) log R)` | `O(1)` thêm | `R = 10^9` |
| 4. Hai con trỏ | `O(M log M + H log H)` | `O(1)` thêm | quét `O(M + H)` |

Với `M = H = 3 × 10^4`: sắp xếp khoảng `4.5 × 10^5` phép so sánh; quét hai con trỏ chỉ `6 × 10^4`.

## 12. Bộ test

Dùng **Cách 1 (vét cạn)** làm oracle độc lập; đối chiếu Cách 2, 3, 4 và `Solution`. Khi gọi, truyền **bản sao** của danh sách vì `Solution` sắp xếp tại chỗ.

In [16]:
cac_cach = {
    "bisect":   find_radius_bisect,
    "binary":   find_radius_binary,
    "twoptr":   find_radius_two_pointers,
    "Solution": lambda h, e: Solution().findRadius(list(h), list(e)),
}

def so_het(houses, heaters, oracle=None):
    if oracle is None:
        oracle = find_radius_brute(list(houses), list(heaters))
    for ten, f in cac_cach.items():
        v = f(list(houses), list(heaters))
        assert v == oracle, (ten, houses, heaters, v, oracle)
    return oracle

for houses, heaters, ans in examples:
    assert so_het(houses, heaters, ans) == ans
print("Vi du de OK")

Vi du de OK


In [17]:
# Edge case
tests = [
    ([1], [1], 0),                                   # nhà trùng lò
    ([1], [10**9], 10**9 - 1),                       # xa nhất có thể
    ([10**9], [1], 10**9 - 1),
    ([1, 1, 1], [1], 0),                             # nhiều nhà trùng nhau
    ([1, 2, 3, 4, 5], [3], 2),                       # một lò giữa
    ([1, 2, 3, 4, 5], [1], 4),                       # lò ở đầu
    ([1, 2, 3, 4, 5], [5], 4),                       # lò ở cuối
    ([5, 4, 3, 2, 1], [3], 2),                       # nhà chưa sắp xếp
    ([1, 2, 3, 4], [4, 1], 1),                       # lò chưa sắp xếp
    ([1, 5, 9], [5], 4),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 10], 4),   # nhà 5, 6 là xa nhất: min(4,5), min(5,4) = 4
    ([282475249, 622650073, 984943658, 144108930, 470211272, 101027544, 457850878, 458777923], [823564440, 115438165, 784484492, 74243042, 114807987, 137522503, 441282327, 16531729, 823378840, 143542612], 161834419),
    ([1, 3], [2, 2, 2], 1),                          # lò trùng nhau
]
for houses, heaters, ans in tests:
    oracle = find_radius_brute(list(houses), list(heaters))
    assert oracle == ans, (houses, heaters, oracle, ans)
    so_het(houses, heaters, oracle)
print("Edge case OK:", len(tests), "test")

Edge case OK: 13 test


**Phản ví dụ** cho các cách hiểu sai thường gặp: `houses = [1,2,3,4]`, `heaters = [1,4]` có đáp án `1`, **không phải** `3` (khoảng cách từ lò `1` tới lò `4`) hay `2` (một nửa khoảng cách): chỉ `max` của "gần nhất" mới đúng. Mục 13 cho thêm các phản ví dụ chạy được.

In [18]:
# Đối chiếu ngẫu nhiên có seed: input nhỏ so với vét cạn
rng = random.Random(475)
for _ in range(1000):
    M, H = rng.randint(1, 8), rng.randint(1, 5)
    hi = rng.choice([5, 10, 30, 10**9])
    houses = [rng.randint(1, hi) for _ in range(M)]
    heaters = [rng.randint(1, hi) for _ in range(H)]
    so_het(houses, heaters)
print("1000 test ngau nhien (nho) khop voi vet can")

1000 test ngau nhien (nho) khop voi vet can


In [19]:
# Input cỡ vừa: vét cạn vẫn chạy được (<= 600 x 600)
rng = random.Random(4750)
for _ in range(60):
    M, H = rng.randint(1, 600), rng.randint(1, 600)
    hi = rng.choice([50, 1000, 10**9])
    houses = [rng.randint(1, hi) for _ in range(M)]
    heaters = [rng.randint(1, hi) for _ in range(H)]
    so_het(houses, heaters)
print("60 test co vua (<= 600 x 600) khop voi vet can")

# Input lớn theo đúng ràng buộc: so 3 cách nhanh với nhau (vét cạn quá chậm)
rng = random.Random(4751)
houses = [rng.randint(1, 10**9) for _ in range(30000)]
heaters = [rng.randint(1, 10**9) for _ in range(30000)]
t0 = time.perf_counter()
kq = {ten: f(list(houses), list(heaters)) for ten, f in cac_cach.items()}
print(f"3 * 10^4 nha va lo: {kq}  ({time.perf_counter() - t0:.2f}s)")
assert len(set(kq.values())) == 1

# Ca lệch: rất nhiều nhà, ít lò và ngược lại
houses = [rng.randint(1, 10**9) for _ in range(30000)]
heaters = [rng.randint(1, 10**9)]
kq = {ten: f(list(houses), list(heaters)) for ten, f in cac_cach.items()}
assert len(set(kq.values())) == 1 and kq["Solution"] == find_radius_brute(houses, heaters)
houses = [rng.randint(1, 10**9)]
heaters = [rng.randint(1, 10**9) for _ in range(30000)]
kq = {ten: f(list(houses), list(heaters)) for ten, f in cac_cach.items()}
assert len(set(kq.values())) == 1 and kq["Solution"] == find_radius_brute(houses, heaters)
print("Ca lech (30000 nha / 1 lo va nguoc lai) OK")

60 test co vua (<= 600 x 600) khop voi vet can


3 * 10^4 nha va lo: {'bisect': 157032, 'binary': 157032, 'twoptr': 157032, 'Solution': 157032}  (0.33s)


Ca lech (30000 nha / 1 lo va nguoc lai) OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp `heaters` trước khi dùng `bisect`

`bisect_left` giả sử mảng đã sắp xếp; trên mảng chưa sắp xếp nó vẫn chạy nhưng cho chỉ số vô nghĩa.

In [20]:
def sai_quen_sort(houses, heaters):
    H = len(heaters)
    INF = float("inf")
    kq = 0
    for x in houses:                                   # SAI: heaters chưa sort
        i = bisect.bisect_left(heaters, x)
        phai = heaters[i] - x if i < H else INF
        trai = x - heaters[i - 1] if i > 0 else INF
        kq = max(kq, min(phai, trai))
    return kq

print("sai :", sai_quen_sort([1, 2, 3, 4], [4, 1]), "| dung:", Solution().findRadius([1, 2, 3, 4], [4, 1]))

sai : 3 | dung: 1


**Vì sao sai:** với `heaters = [4, 1]` (chưa sắp xếp), `bisect_left` đoán sai vị trí chèn, nên hai ứng viên gần nhất không phải là hai lò kề bên nhà.

**Code đúng:** `heaters.sort()` (hoặc `sorted`) trước.

### Lỗi 2 — Chỉ xét lò bên phải (hoặc chỉ bên trái)

Nhiều người viết: "lò gần nhất là `heaters[bisect_left(...)]`" mà quên rằng lò **bên trái** (`i - 1`) có thể gần hơn.

In [21]:
def sai_mot_phia(houses, heaters):
    heaters = sorted(heaters)
    H = len(heaters)
    kq = 0
    for x in houses:
        i = bisect.bisect_left(heaters, x)
        i = min(i, H - 1)                              # SAI: chỉ xét lò tại i (bên phải), bỏ lò bên trái
        kq = max(kq, abs(heaters[i] - x))
    return kq

print("sai :", sai_mot_phia([1, 2, 3, 4], [1, 4]), "| dung:", Solution().findRadius([1, 2, 3, 4], [1, 4]))
print("sai :", sai_mot_phia([1, 2, 3, 4, 5], [1, 5]), "| dung:", Solution().findRadius([1, 2, 3, 4, 5], [1, 5]))

sai : 2 | dung: 1
sai : 3 | dung: 2


**Vì sao sai:** nhà `2` có `bisect_left([1,4], 2) = 1`, lò phải là `4` (cách `2`) nhưng lò trái `1` cách chỉ `1`. Bản sai trả `2` thay vì `1`.

**Code đúng:** xét **cả** `heaters[i]` và `heaters[i - 1]` (mục 7, Cách 2), nhớ xử lý khi `i == 0` hoặc `i == H`.

### Lỗi 3 — Lấy `min` thay vì `max` trên các nhà (hoặc nhầm thứ tự `min`/`max`)

Công thức là `max` qua nhà của `min` qua lò. Đảo thành `min` qua nhà sẽ chỉ phủ nhà dễ nhất.

In [22]:
def sai_min_min(houses, heaters):
    return min(min(abs(x - h) for h in heaters) for x in houses)      # SAI: min bên ngoài

def sai_max_max(houses, heaters):
    return max(max(abs(x - h) for h in heaters) for x in houses)      # SAI: max bên trong

h, e = [1, 5], [2]
print("sai min-min:", sai_min_min(h, e), "| sai max-max:", sai_max_max(h, e), "| dung:", Solution().findRadius(list(h), list(e)))
h, e = [1, 2, 3, 4], [1, 4]
print("sai min-min:", sai_min_min(h, e), "| sai max-max:", sai_max_max(h, e), "| dung:", Solution().findRadius(list(h), list(e)))

sai min-min: 1 | sai max-max: 3 | dung: 3
sai min-min: 0 | sai max-max: 3 | dung: 1


**Vì sao sai:** `min` bên ngoài chỉ đảm bảo **một** nhà được phủ; `max` bên trong buộc **mỗi nhà** phủ bởi **mọi** lò (không cần thiết, một lò là đủ). Chỉ `max(min)` thỏa "mọi nhà đều có một lò đủ gần".

### Lỗi 4 — Cận trên của nhị phân quá nhỏ

Nếu đặt `right = max(houses)` (hoặc `len(houses)`) cho nhị phân trên `r`, đáp án bị chặn sai khi lò ở xa nhà.

In [23]:
def sai_can_tren(houses, heaters):
    houses = sorted(houses)
    heaters = sorted(heaters)
    M, H = len(houses), len(heaters)

    def duoc(r):
        i = j = 0
        while i < M:
            if j >= H:
                return False
            lo, hi = heaters[j] - r, heaters[j] + r
            if houses[i] < lo:
                return False
            if houses[i] > hi:
                j += 1
            else:
                i += 1
        return True

    left, right = 0, max(houses)                       # SAI: cận trên quá nhỏ
    while left < right:
        mid = (left + right) // 2
        if duoc(mid):
            right = mid
        else:
            left = mid + 1
    return left

print("sai :", sai_can_tren([1], [10**9]), "| dung:", Solution().findRadius([1], [10**9]))
print("sai :", sai_can_tren([1, 2], [100]), "| dung:", Solution().findRadius([1, 2], [100]))

sai : 1 | dung: 999999999
sai : 2 | dung: 99


**Vì sao sai:** nhà `1` và lò `10^9` cần `r = 10^9 - 1`, nhưng cận trên `max(houses) = 1` làm nhị phân dừng ở `1` mà `duoc(1)` là `False`. Vòng lặp trả về `left = right = 1` mà **không kiểm tra** `duoc(right)`.

**Code đúng:** cận trên phải đảm bảo `duoc(right) = True`; dùng `10^9` (hoặc `max(max(houses), max(heaters)) - min(min(houses), min(heaters))`).

### Lỗi 5 — Quên rằng nhà nằm ngoài cùng cần "chỉ một phía"

Khi `i == 0` thì **không có** lò trái, `i == H` thì **không có** lò phải. Truy cập `heaters[-1]` (Python cho chỉ số âm!) sẽ lấy phần tử **cuối** một cách im lặng.

In [24]:
def sai_chi_so_am(houses, heaters):
    heaters = sorted(heaters)
    H = len(heaters)
    kq = 0
    for x in houses:
        i = bisect.bisect_left(heaters, x)
        phai = heaters[i] - x if i < H else float("inf")
        trai = x - heaters[i - 1]                      # SAI: khi i == 0, heaters[-1] là lò CUỐI -> d âm
        kq = max(kq, min(phai, trai))
    return kq

print("sai :", sai_chi_so_am([1], [5, 9]), "| dung:", Solution().findRadius([1], [5, 9]))
print("sai :", sai_chi_so_am([1, 2], [3, 7]), "| dung:", Solution().findRadius([1, 2], [3, 7]))

sai : 0 | dung: 4
sai : 0 | dung: 2


**Vì sao sai:** với nhà `1` và `heaters = [5, 9]`: `i = 0`, `heaters[i - 1] = heaters[-1] = 9` (lò cuối) và `x - 9 = -8` là số **âm**; `min(phai, trai)` chọn số âm nên đáp án bị tính thấp hơn thực tế (`0` thay vì `4`). Luôn bọc `if i > 0`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mỗi điểm cần phục vụ bởi điểm gần nhất trong tập khác, tối thiểu hóa khoảng cách xa nhất" → công thức `max(min)`, thường giải bằng **sắp xếp + bisect** hoặc **hai con trỏ**.
- **Phần tử gần nhất trong mảng đã sắp xếp:** `bisect_left` rồi so `arr[i - 1]` và `arr[i]`; nhớ xử lý biên.
- **"Nhỏ nhất sao cho thỏa" + đơn điệu → nhị phân trên đáp án.** Viết hàm `duoc(r)`; kiểm tra `duoc(right)` luôn `True`.
- **Hai mảng đều sắp xếp** → thử hai con trỏ chỉ tiến, ra `O(M + H)`.
- **Bài họ hàng:** LeetCode 744 *Find Smallest Letter Greater Than Target* (bisect), 1011 *Capacity To Ship Packages Within D Days* (nhị phân trên đáp án), 1642 *Furthest Building You Can Reach*, 2594 *Minimum Time to Repair Cars*, 2517 *Maximum Tastiness of Candy Basket*.

## 15. Biến thể

1. **Mỗi lò có bán kính riêng cho trước**: bài trở thành kiểm tra phủ (sắp xếp khoảng phủ, quét).
2. **Chỉ có `k` lò, chọn vị trí**: tìm `r` nhỏ nhất (bài lớp "k-center trên đường thẳng"): nhị phân trên `r` + tham lam đặt lò.
3. **Trả về lò phục vụ từng nhà** thay vì bán kính: lưu chỉ số `j` trong Cách 4.
4. **Nhà và lò trên đường tròn** (vòng): xét thêm khoảng cách vòng quanh.
5. **Bán kính nguyên vs thực:** trên đường thẳng đáp án luôn nguyên khi vị trí nguyên (vì là khoảng cách giữa hai số nguyên).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn | `max` nhà của `min` lò | `O(M H)` | `O(1)` | oracle, input rất nhỏ |
| 2. Sắp xếp + bisect | lò gần nhất cạnh vị trí chèn | `O(H log H + M log H)` | `O(1)` | nhà không cần sắp xếp, dễ nhớ |
| 3. Nhị phân trên `r` | `duoc(r)` đơn điệu, hai con trỏ | `O(... + (M+H) log R)` | `O(1)` | khi cần thuật toán kiểu "tìm đáp án nhỏ nhất thỏa" |
| 4. Hai con trỏ | lò gần nhất chỉ tiến phải | `O(M log M + H log H)` | `O(1)` | **nộp bài**, nhanh và gọn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `bisect.bisect_left([1, 4, 9], 5)`; (b) `bisect.bisect_left([1, 4, 9], 4)`; (c) `max(min(abs(x - h) for h in [3]) for x in [1, 3, 8])`; (d) `Solution().findRadius([1], [1])`; (e) `Solution().findRadius([1, 5], [3])`.

**Bài 2 (tính tay).** Với `houses = [2, 6, 7, 12]`, `heaters = [4, 10]`: lập bảng nhà → lò gần nhất → khoảng cách, rồi nêu đáp án.

**Bài 3 (điền TODO).** Hoàn thành `lo_gan_nhat(heaters, x)` trả về **khoảng cách** từ `x` tới lò gần nhất trong mảng `heaters` **đã sắp xếp**, dùng `bisect_left`.

In [25]:
def lo_gan_nhat(heaters, x):
    # TODO 1: i = bisect.bisect_left(heaters, x)
    # TODO 2: ứng viên phải: heaters[i] - x nếu i < len(heaters)
    # TODO 3: ứng viên trái: x - heaters[i - 1] nếu i > 0
    # TODO 4: trả về min của các ứng viên
    return None

# KHÔNG gọi lo_gan_nhat ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `find_radius_todo(houses, heaters)` theo Cách 3 (nhị phân trên `r` + hai con trỏ) **không nhìn lại** mục 7. Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Mỗi lò sưởi có thể có **bán kính khác nhau** nhưng ta được phép chọn một **hệ số** `c >= 0` (nguyên) sao cho lò `j` có bán kính `c * w[j]` với `w[j]` cho trước (`w[j] >= 1`). Hãy viết `find_c(houses, heaters, w)` trả về `c` nhỏ nhất để phủ mọi nhà. Kiểm tra: `find_c([1, 5], [2], [1]) == 3` (giống đề) và `find_c([1, 2, 3, 4], [1, 4], [1, 1]) == 1`. Gợi ý: nhị phân trên `c`; hàm `duoc(c)` kiểm tra từng nhà có lò `j` với `|x - h_j| <= c * w[j]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `2` (vì `9 >= 5`, chỉ số `2`); (b) `1`; (c) `5` (nhà `8` cách `3` là `5`); (d) `0`; (e) `2`. Bài 3: nhớ biên `i == 0` và `i == len`.
- **Mức 2 (vừa):** Bài 2: lò `[4, 10]`: nhà `2` → `2`; nhà `6` → `min(2, 4) = 2`; nhà `7` → `min(3, 3) = 3`; nhà `12` → `2`. Đáp án `3`. Bài 4: `duoc(r)` trả `False` khi `houses[i] < heaters[j] - r`.
- **Mức 3 (sâu):** Bài 5: `duoc(c)` có thể dùng vét cạn `O(M * H)` trong bài tập (input nhỏ); sau đó thử tối ưu bằng cách nhận xét: vì bán kính khác nhau nên hai con trỏ **không còn đúng**, cần quét khoảng phủ `[h - c w, h + c w]` rồi hợp các khoảng.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
assert bisect.bisect_left([1, 4, 9], 5) == 2
assert bisect.bisect_left([1, 4, 9], 4) == 1
assert max(min(abs(x - h) for h in [3]) for x in [1, 3, 8]) == 5
assert Solution().findRadius([1], [1]) == 0
assert Solution().findRadius([1, 5], [3]) == 2
print("Bài 1 OK")

# Bài 2
houses, heaters = [2, 6, 7, 12], [4, 10]
bang = [(x, min(heaters, key=lambda h: abs(h - x)), min(abs(x - h) for h in heaters)) for x in houses]
print("Bài 2:", bang)
assert [d for _, _, d in bang] == [2, 2, 3, 2]
assert Solution().findRadius(list(houses), list(heaters)) == 3
print("Bài 2 OK")

# Bài 3
def lo_gan_nhat_ans(heaters, x):
    i = bisect.bisect_left(heaters, x)
    ung_vien = []
    if i < len(heaters):
        ung_vien.append(heaters[i] - x)
    if i > 0:
        ung_vien.append(x - heaters[i - 1])
    return min(ung_vien)

assert [lo_gan_nhat_ans([2, 10], x) for x in [1, 2, 6, 7, 12]] == [1, 0, 4, 3, 2]
assert lo_gan_nhat_ans([5], 5) == 0
print("Bài 3 OK")

# Bài 4
def find_radius_todo_ans(houses, heaters):
    houses = sorted(houses)
    heaters = sorted(heaters)
    def duoc(r):
        i = j = 0
        while i < len(houses):
            if j >= len(heaters):
                return False
            if houses[i] < heaters[j] - r:
                return False
            if houses[i] > heaters[j] + r:
                j += 1
            else:
                i += 1
        return True
    lo, hi = 0, 10**9
    while lo < hi:
        mid = (lo + hi) // 2
        if duoc(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for h, e, ans in examples:
    assert find_radius_todo_ans(h, e) == ans
random.seed(45)
for _ in range(300):
    h = [random.randint(1, 40) for _ in range(random.randint(1, 8))]
    e = [random.randint(1, 40) for _ in range(random.randint(1, 5))]
    assert find_radius_todo_ans(h, e) == find_radius_brute(h, e)
print("Bài 4 OK")

# Bài 5
def find_c(houses, heaters, w):
    def duoc(c):
        return all(any(abs(x - h) <= c * ww for h, ww in zip(heaters, w)) for x in houses)
    lo, hi = 0, 10**9
    while lo < hi:
        mid = (lo + hi) // 2
        if duoc(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

assert find_c([1, 5], [2], [1]) == 3
assert find_c([1, 2, 3, 4], [1, 4], [1, 1]) == 1
assert find_c([1, 9], [2, 8], [1, 10]) == 1
assert find_c([1, 20], [2, 8], [1, 1]) == 12
assert find_c([5, 6, 7], [6], [3]) == 1
random.seed(46)
for _ in range(200):
    h = [random.randint(1, 30) for _ in range(random.randint(1, 6))]
    e = [random.randint(1, 30) for _ in range(random.randint(1, 4))]
    assert find_c(h, e, [1] * len(e)) == find_radius_brute(h, e)
print("Bài 5 OK")

Bài 1 OK
Bài 2: [(2, 4, 2), (6, 4, 2), (7, 4, 3), (12, 10, 2)]
Bài 2 OK
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (hai con trỏ, nhà & lò sắp xếp):**

```python
houses.sort(); heaters.sort()
j = 0
kq = 0
for x in houses:
    while j + 1 < len(heaters) and abs(heaters[j + 1] - x) <= abs(heaters[j] - x):
        j += 1
    kq = max(kq, abs(heaters[j] - x))
return kq
```

**Template (bisect, chỉ cần sắp xếp lò):**

```python
heaters.sort()
for x in houses:
    i = bisect.bisect_left(heaters, x)
    d = min(heaters[i] - x if i < H else INF, x - heaters[i - 1] if i > 0 else INF)
    kq = max(kq, d)
```

**Checklist:**

- [ ] Đáp án là `max` qua nhà của `min` qua lò (không phải `min`/`sum`).
- [ ] Đã **sắp xếp** `heaters` (và `houses` nếu dùng hai con trỏ / nhị phân với hai con trỏ).
- [ ] Với `bisect`: xét **cả hai phía** (`i` và `i - 1`) và xử lý biên `i == 0`, `i == H` (cẩn thận chỉ số âm).
- [ ] Nếu nhị phân trên `r`: cận trên đảm bảo `duoc(right) = True` (`10^9`); điều kiện cập nhật `right = mid` / `left = mid + 1`.
- [ ] Khi test nhiều cách, truyền **bản sao** vì `sort()` đổi mảng đầu vào.
- [ ] Thử tay `[1,2,3]/[2]` (1), `[1,2,3,4]/[1,4]` (1), `[1,5]/[2]` (3), `[1]/[10**9]` (`10**9 - 1`).
- [ ] Đúng signature `class Solution` / `findRadius(self, houses: List[int], heaters: List[int]) -> int`; độ phức tạp `O(M log M + H log H)` thời gian, `O(1)` thêm bộ nhớ.